# 🏥 의료 데이터 분석
## 02. 피처 엔지니어링 (Feature Engineering)

---

이 노트북에서는:
1. 데이터 타입 변환 (날짜, 범주형, 숫자형)
2. 데이터 병합 (merge)
3. 파생 변수 생성
4. 최종 분석용 데이터셋 구축

---
## 1. 환경 설정 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# 데이터 URL
URLS = {
    "patients": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/PatientCorePopulatedTable.txt",
    "admissions": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/AdmissionsCorePopulatedTable.txt",
    "diagnoses": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/AdmissionsDiagnosesCorePopulatedTable.txt",
    "labs": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/LabsCorePopulatedTable.txt"
}

def load_data(url, columns):
    response = requests.get(url)
    df = pd.read_csv(StringIO(response.text), sep='\t', skiprows=1, header=None, names=columns)
    return df

# 데이터 로드
df_patients = load_data(
    URLS['patients'],
    ['PatientID', 'PatientGender', 'PatientDateOfBirth', 'PatientRace', 
     'PatientMaritalStatus', 'PatientLanguage', 'PatientPopulationPercentageBelowPoverty']
)

df_admissions = load_data(
    URLS['admissions'],
    ['PatientID', 'AdmissionID', 'AdmissionStartDate', 'AdmissionEndDate']
)

df_diagnoses = load_data(
    URLS['diagnoses'],
    ['PatientID', 'AdmissionID', 'PrimaryDiagnosisCode', 'PrimaryDiagnosisDescription']
)

df_labs = load_data(
    URLS['labs'],
    ['PatientID', 'AdmissionID', 'LabName', 'LabValue', 'LabUnits', 'LabDateTime']
)

print("✅ 데이터 로드 완료")

✅ 데이터 로드 완료


---
## 2. 데이터 타입 변환

분석에 적합한 데이터 타입으로 변환합니다.

### 2.1 날짜형 변환

In [2]:
# Patients: 생년월일
df_patients['PatientDateOfBirth'] = pd.to_datetime(df_patients['PatientDateOfBirth'], errors='coerce')

# Admissions: 입원 시작/종료일
df_admissions['AdmissionStartDate'] = pd.to_datetime(df_admissions['AdmissionStartDate'], errors='coerce')
df_admissions['AdmissionEndDate'] = pd.to_datetime(df_admissions['AdmissionEndDate'], errors='coerce')

# Labs: 검사 날짜
df_labs['LabDateTime'] = pd.to_datetime(df_labs['LabDateTime'], errors='coerce')

print("✅ 날짜형 변환 완료")

✅ 날짜형 변환 완료


### 2.2 범주형 변환

In [3]:
# Patients
df_patients['PatientGender'] = df_patients['PatientGender'].astype('category')
df_patients['PatientRace'] = df_patients['PatientRace'].astype('category')
df_patients['PatientMaritalStatus'] = df_patients['PatientMaritalStatus'].astype('category')
df_patients['PatientLanguage'] = df_patients['PatientLanguage'].astype('category')

# Admissions  
df_admissions['AdmissionID'] = df_admissions['AdmissionID'].astype('category')

# Diagnoses
df_diagnoses['AdmissionID'] = df_diagnoses['AdmissionID'].astype('category')
df_diagnoses['PrimaryDiagnosisCode'] = df_diagnoses['PrimaryDiagnosisCode'].astype('category')

# Labs
df_labs['AdmissionID'] = df_labs['AdmissionID'].astype('category')
df_labs['LabName'] = df_labs['LabName'].astype('category')

print("✅ 범주형 변환 완료")

✅ 범주형 변환 완료


### 2.3 숫자형 변환

In [4]:
# Patients: 빈곤율
df_patients['PatientPopulationPercentageBelowPoverty'] = pd.to_numeric(
    df_patients['PatientPopulationPercentageBelowPoverty'], errors='coerce'
)

# Labs: 검사 값
df_labs['LabValue'] = pd.to_numeric(df_labs['LabValue'], errors='coerce')

print("✅ 숫자형 변환 완료")
print("\n변환 후 데이터 타입:")
print(df_patients.dtypes)

✅ 숫자형 변환 완료

변환 후 데이터 타입:
PatientID                                          object
PatientGender                                    category
PatientDateOfBirth                         datetime64[ns]
PatientRace                                      category
PatientMaritalStatus                             category
PatientLanguage                                  category
PatientPopulationPercentageBelowPoverty           float64
dtype: object


---
## 3. 파생 변수 생성

분석에 유용한 새로운 변수들을 만듭니다.

### 💡 파생변수 생성 목적 및 근거
데이터의 원천 정보를 결합하여 모델이 질병의 중증도와 입원 패턴을 더 잘 학습할 수 있도록 아래와 같은 파생변수를 생성합니다.

* **Age_Group:** 연령은 건강 상태의 주요 결정 요인이며, 10~20년 단위의 그룹화는 의료 정책 및 임상 통계에서 환자군을 분류하는 표준적인 방식입니다.
* **LengthOfStay (LOS):** 입원 기간은 병원 운영 효율성과 환자의 중증도를 나타내는 핵심 지표로, 퇴원일과 입원일의 차이를 통해 계산합니다.
* **Total_Abnormal_Count:** 단일 검사 수치보다 '정상 범위를 벗어난 총 횟수'가 환자의 전반적인 생체 징후 불안정성을 더 직접적으로 반영합니다.
* **Lab_Trend:** 수치의 단순 평균뿐만 아니라 입원 중 수치가 호전되는지 악화되는지의 방향성을 파악하기 위해 생성합니다.

### 3.1 환자 관련 파생 변수

In [5]:
# 나이 계산
df_patients['Age'] = (pd.Timestamp.now() - df_patients['PatientDateOfBirth']).dt.days / 365.25
df_patients['Age'] = df_patients['Age'].round(1)

# 연령대 그룹
df_patients['AgeGroup'] = pd.cut(
    df_patients['Age'],
    bins=[0, 30, 40, 50, 60, 70, 200],
    labels=['~30세', '30대', '40대', '50대', '60대', '70세~']
)

# 빈곤 수준 카테고리
df_patients['PovertyLevel'] = pd.cut(
    df_patients['PatientPopulationPercentageBelowPoverty'],
    bins=[0, 10, 20, 30, 100],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print("✅ 환자 파생 변수 생성 완료")
print(f"\n연령대 분포:\n{df_patients['AgeGroup'].value_counts().sort_index()}")

✅ 환자 파생 변수 생성 완료

연령대 분포:
AgeGroup
~30세     0
30대      4
40대     13
50대     16
60대     20
70세~    47
Name: count, dtype: int64


### 3.2 입원 관련 파생 변수

In [6]:
# 입원 기간 (일)
df_admissions['LengthOfStay'] = (
    df_admissions['AdmissionEndDate'] - df_admissions['AdmissionStartDate']
).dt.days

# 입원 연도
df_admissions['AdmissionYear'] = df_admissions['AdmissionStartDate'].dt.year

# 입원 월
df_admissions['AdmissionMonth'] = df_admissions['AdmissionStartDate'].dt.month

# 입원 요일
df_admissions['AdmissionDayOfWeek'] = df_admissions['AdmissionStartDate'].dt.dayofweek
weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df_admissions['AdmissionDayName'] = df_admissions['AdmissionDayOfWeek'].map(weekday_map)

# 입원 기간 카테고리
df_admissions['LengthOfStayCategory'] = pd.cut(
    df_admissions['LengthOfStay'],
    bins=[-1, 3, 7, 14, 30, 1000],
    labels=['초단기(~3일)', '단기(4-7일)', '중기(8-14일)', '장기(15-30일)', '초장기(31일~)']
)

print("✅ 입원 파생 변수 생성 완료")
print(f"\n입원 기간 통계:\n평균: {df_admissions['LengthOfStay'].mean():.1f}일")
print(f"중앙값: {df_admissions['LengthOfStay'].median():.1f}일")

✅ 입원 파생 변수 생성 완료

입원 기간 통계:
평균: 10.6일
중앙값: 10.0일


### 3.3 검사 관련 집계 변수

In [7]:
# 1. 이상치 판별을 위한 기준 통계량 생성 (LabName별)
df_lab_thresholds = df_labs.groupby('LabName')['LabValue'].agg(['mean', 'std']).reset_index()
df_lab_thresholds['upper'] = df_lab_thresholds['mean'] + 1.5 * df_lab_thresholds['std']
df_lab_thresholds['lower'] = df_lab_thresholds['mean'] - 1.5 * df_lab_thresholds['std']

# 2. 원본 데이터에 기준 결합 및 이상치 판별 (IsAbnormal 컬럼 추가)
df_labs = df_labs.merge(df_lab_thresholds[['LabName', 'upper', 'lower']], on='LabName', how='left')
df_labs['IsAbnormal'] = ((df_labs['LabValue'] > df_labs['upper']) | (df_labs['LabValue'] < df_labs['lower'])).astype(int)

# 3. 입원별 검사 횟수 및 종류 집계
df_lab_counts = df_labs.groupby(['PatientID', 'AdmissionID']).size().reset_index(name='LabTestCount')
df_lab_variety = df_labs.groupby(['PatientID', 'AdmissionID'])['LabName'].nunique().reset_index(name='LabTestVariety')

# 4. 입원 + 검사종류별 상세 집계 (Mean, Std, Trend 등)
df_lab_features = df_labs.groupby(['PatientID', 'AdmissionID', 'LabName']).agg(
    Lab_Mean=('LabValue', 'mean'),
    Lab_Std=('LabValue', 'std'),
    Abnormal_Sum=('IsAbnormal', 'sum'),
    First_Value=('LabValue', 'first'),
    Last_Value=('LabValue', 'last')
).reset_index()

# 추이(Trend) 계산
df_lab_features['Lab_Trend'] = df_lab_features['Last_Value'] - df_lab_features['First_Value']

# 5. 결과 출력
print("✅ 검사 집계 변수 생성 및 네이밍 정리 완료")
print("-" * 40)
print(f"입원당 평균 검사 횟수: {df_lab_counts['LabTestCount'].mean():.1f}회")
print(f"입원당 평균 검사 종류: {df_lab_variety['LabTestVariety'].mean():.1f}종")

print("\n🔍 검사 지표 심층 분석")
total_records = len(df_labs)
abnormal_records = df_labs['IsAbnormal'].sum()
print(f"✓ 전체 검사 건수 중 이상치(Abnormal) 비율: {(abnormal_records/total_records)*100:.1f}%")

improving = (df_lab_features['Lab_Trend'] < 0).sum()
worsening = (df_lab_features['Lab_Trend'] > 0).sum()
print(f"✓ 검사 수치 감소(Trend < 0) 케이스: {improving}건")
print(f"✓ 검사 수치 증가(Trend > 0) 케이스: {worsening}건")

✅ 검사 집계 변수 생성 및 네이밍 정리 완료
----------------------------------------
입원당 평균 검사 횟수: 159.3회
입원당 평균 검사 종류: 18.6종

🔍 검사 지표 심층 분석
✓ 전체 검사 건수 중 이상치(Abnormal) 비율: 12.8%
✓ 검사 수치 감소(Trend < 0) 케이스: 6049건
✓ 검사 수치 증가(Trend > 0) 케이스: 5984건


---
## 4. 데이터 병합 (Merge)

검사 집계 변수가 추가된 검사 테이블은 Wide 포맷으로 변환합니다.
분석용 통합 데이터셋을 만듭니다.

In [8]:
# 1. Wide 포맷 변환 (Lab 상세 지표)
master_wide = df_lab_features.pivot_table(
    index=['PatientID', 'AdmissionID'],
    columns='LabName',
    values=['Lab_Mean', 'Lab_Std', 'Abnormal_Sum', 'Lab_Trend']
)
master_wide.columns = [f"{col[0]}_{col[1]}" for col in master_wide.columns]
master_wide = master_wide.reset_index()

# 2. 메인 데이터 병합 (Step by Step)
# Step 1: Admissions + Patients
df_main = df_admissions.merge(df_patients, on='PatientID', how='left')

# Step 2: + Diagnoses
df_main = df_main.merge(df_diagnoses, on=['PatientID', 'AdmissionID'], how='left')

# Step 3: + Lab 상세 지표 (Wide format)
df_main = df_main.merge(master_wide, on=['PatientID', 'AdmissionID'], how='inner')

# Step 4: + Lab 횟수 및 종류 (누락되었던 부분 추가)
df_main = df_main.merge(df_lab_counts, on=['PatientID', 'AdmissionID'], how='left')
df_main = df_main.merge(df_lab_variety, on=['PatientID', 'AdmissionID'], how='left')

# 3. 결측치 처리 및 중증도 지표 생성
# 검사 안 한 경우 0으로 채우기
df_main['LabTestCount'] = df_main['LabTestCount'].fillna(0).astype(int)
df_main['LabTestVariety'] = df_main['LabTestVariety'].fillna(0).astype(int)

# 통합 중증도 지표 (변수명을 df_main으로 통일)
df_main['Total_Abnormal_Count'] = df_main.filter(like='Abnormal_Sum').sum(axis=1)
df_main['Total_Lab_Variability'] = df_main.filter(like='Lab_Std').mean(axis=1)

# 4. 결과 출력
print("✅ 데이터 병합 및 지표 생성 완료")
print(f"최종 데이터셋 크기: {df_main.shape}")
print("-" * 40)

# 2. 환자별 중증도 분포
print(f"✓ 입원당 평균 이상치 발생 건수: {df_main['Total_Abnormal_Count'].mean():.1f}건")
print(f"✓ 최대 이상치 발생 건수(최고 위험군): {df_main['Total_Abnormal_Count'].max()}건")

# 3. 가변성(Variability) 지표 확인
print(f"✓ 전체 환자 평균 Lab 가변성(Std): {df_main['Total_Lab_Variability'].mean():.2f}")

# 5. 데이터 누락 확인
missing_variability = df_main['Total_Lab_Variability'].isna().sum()
if missing_variability > 0:
    print(f"⚠️ 주의: 검사 기록이 1회뿐이라 가변성(Std)을 계산할 수 없는 입원 건: {missing_variability}건")

✅ 데이터 병합 및 지표 생성 완료
최종 데이터셋 크기: (372, 165)
----------------------------------------
✓ 입원당 평균 이상치 발생 건수: 38.5건
✓ 최대 이상치 발생 건수(최고 위험군): 86건
✓ 전체 환자 평균 Lab 가변성(Std): 7.37


---
## 5. 최종 데이터 검증

In [9]:
print("📊 [Advanced Data Validation]")

# 1. 논리적 타당성 검사 (Sanity Check)
invalid_los = df_main[df_main['LengthOfStay'] <= 0]
invalid_age = df_main[df_main['Age'] < 0]

if len(invalid_los) == 0 and len(invalid_age) == 0:
    print("✅ 논리적 오류 없음 (입원기간 및 나이 수치 정상)")
else:
    print(f"⚠️ 경고: 입원기간 오류 {len(invalid_los)}건, 나이 오류 {len(invalid_age)}건 발견")

# 2. 데이터 규모 및 고유 환자 수 확인
unique_patients = df_main['PatientID'].nunique()
total_admissions = len(df_main)
print(f"✅ 분석 대상: 총 {unique_patients}명의 환자, {total_admissions}건의 입원 사례")

# 3. 타겟 변수(LOS) 기초 통계
print("\n🎯 타겟 변수(LengthOfStay) 분포 요약:")
print(df_main['LengthOfStay'].describe()[['mean', '50%', 'max', 'min']].to_string())

# 4. 주요 피처 생성 확인 (Total_Lab_Variability 등)
if 'Total_Lab_Variability' in df_main.columns:
    print(f"\n✅ 가변성 지표 확인 완료 (평균: {df_main['Total_Lab_Variability'].mean():.2f})")


📊 [Advanced Data Validation]
✅ 논리적 오류 없음 (입원기간 및 나이 수치 정상)
✅ 분석 대상: 총 100명의 환자, 372건의 입원 사례

🎯 타겟 변수(LengthOfStay) 분포 요약:
mean    10.556452
50%     10.000000
max     19.000000
min      2.000000

✅ 가변성 지표 확인 완료 (평균: 7.37)


In [10]:
# 변환 전/후 주요 컬럼 비교 테이블 생성
comparison_data = {
    "구분": ["연령 정보", "시간 정보", "검사 수치", "입원 기간", "데이터 구조"],
    "변환 전 (Raw)": ["BirthDate (날짜)", "AdmissionDate (문자열)", "LabValue (문자열/혼합)", "없음", "4개 테이블 분산"],
    "변환 후 (Feature)": ["Age / Age_Group (숫자/범주)", "AdmissionDate (Datetime 객체)", "LabValue (Float)", "LengthOfStay (일수)", "df_main (1개 통합셋)"]
}

df_comparison = pd.DataFrame(comparison_data)

print("### [참고] 데이터 피처 엔지니어링 전/후 비교 ###")
display(df_comparison)

print("\n🚀 데이터셋 준비 완료. 다음 단계(EDA)로 이동 가능합니다.")

### [참고] 데이터 피처 엔지니어링 전/후 비교 ###


,구분,변환 전 (Raw),변환 후 (Feature)
0,연령 정보,BirthDate (날짜),Age / Age_Group (숫자/범주)
1,시간 정보,AdmissionDate (문자열),AdmissionDate (Datetime 객체)
2,검사 수치,LabValue (문자열/혼합),LabValue (Float)
3,입원 기간,없음,LengthOfStay (일수)
4,데이터 구조,4개 테이블 분산,df_main (1개 통합셋)



🚀 데이터셋 준비 완료. 다음 단계(EDA)로 이동 가능합니다.


In [11]:
# 샘플 데이터 확인
df_main.head(10)

,PatientID,AdmissionID,AdmissionStartDate,AdmissionEndDate,LengthOfStay,AdmissionYear,AdmissionMonth,AdmissionDayOfWeek,AdmissionDayName,LengthOfStayCategory,...,Lab_Trend_METABOLIC: SODIUM,Lab_Trend_METABOLIC: TOTAL PROTEIN,Lab_Trend_URINALYSIS: PH,Lab_Trend_URINALYSIS: RED BLOOD CELLS,Lab_Trend_URINALYSIS: SPECIFIC GRAVITY,Lab_Trend_URINALYSIS: WHITE BLOOD CELLS,LabTestCount,LabTestVariety,Total_Abnormal_Count,Total_Lab_Variability
0,7A025E77-7832-4F53-B9A7-09A3F98AC17E,7,2011-10-12 14:55:02.027,2011-10-22 01:16:07.557,9,2011,10,2,수,중기(8-14일),...,15.7,0.4,0.1,-0.3,0.0,3.7,230,35,34,6.985972
1,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,1,1993-02-11 18:57:04.003,1993-02-24 17:22:29.713,12,1993,2,3,목,중기(8-14일),...,-14.1,1.3,-0.5,-1.4,0.0,1.1,355,35,49,8.480702
2,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,2,2002-11-28 19:06:31.117,2002-12-04 19:14:40.797,6,2002,11,3,목,단기(4-7일),...,17.6,-1.8,-1.5,-0.2,0.0,0.3,202,35,28,6.837585
3,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,3,2011-07-19 18:42:45.287,2011-07-25 04:57:42.053,5,2011,7,1,화,단기(4-7일),...,-1.2,0.5,1.0,2.0,0.0,-0.1,150,35,23,8.588358
4,886B5885-1EE2-49F3-98D5-A2F02EB8A9D4,1,1994-12-03 22:20:46.077,1994-12-20 20:24:56.010,16,1994,12,5,토,장기(15-30일),...,-12.3,-2.5,-1.4,0.5,0.0,4.1,424,35,54,8.029381
5,886B5885-1EE2-49F3-98D5-A2F02EB8A9D4,2,2007-10-26 23:43:13.310,2007-11-01 06:54:26.723,5,2007,10,4,금,단기(4-7일),...,-17.5,-0.1,1.9,-0.3,0.0,-1.5,168,35,26,6.538820
6,886B5885-1EE2-49F3-98D5-A2F02EB8A9D4,3,2009-05-24 16:55:08.567,2009-05-26 20:15:49.657,2,2009,5,6,일,초단기(~3일),...,-1.5,0.0,-1.9,0.0,0.0,4.9,77,35,8,5.405974
7,0E0EADE8-5592-4E0B-9F88-D7596E32EE08,1,1970-05-12 08:54:02.500,1970-05-23 21:22:35.027,11,1970,5,1,화,중기(8-14일),...,1.7,-0.2,0.6,-0.4,0.0,-3.3,306,35,39,8.705388
8,0E0EADE8-5592-4E0B-9F88-D7596E32EE08,2,2002-01-09 00:46:30.870,2002-01-12 19:17:15.703,3,2002,1,2,수,초단기(~3일),...,0.1,2.6,-0.3,1.3,0.0,-0.9,108,35,13,7.341436
9,0E0EADE8-5592-4E0B-9F88-D7596E32EE08,3,2008-02-27 05:36:05.627,2008-03-07 01:56:29.463,8,2008,2,2,수,중기(8-14일),...,16.9,3.0,-1.6,1.7,0.0,3.6,239,35,25,8.209629


---
## 6. 데이터 저장

전처리된 데이터를 CSV로 저장할 수 있습니다.

In [12]:
df_main.to_csv('processed_healthcare_data.csv', index=False, encoding='utf-8-sig')
print("✅ 데이터 저장 완료: processed_healthcare_data.csv")

✅ 데이터 저장 완료: processed_healthcare_data.csv


---
## 7. 요약

**완료된 작업:**
- ✅ **날짜형 변환:** Admission, Lab 관련 4개 컬럼 정교화
- ✅ **범주형 변환:** 성별, 인종 등 8개 기본 컬럼 변환
- ✅ **숫자형 변환:** 검사 수치(LabValue) 등 분석용 데이터 수치화
- ✅ **파생 변수 생성:** 13개 이상 (나이, 입원기간, **검사 가변성(Std), 검사 추이(Trend), 이상치 합계** 등)
- ✅ **데이터 병합:** 4개 테이블 → 1개 통합 마스터 데이터셋(`df_main`) 구축

**생성된 주요 검사 지표:**
- **Lab_Std & Total_Lab_Variability:** 환자 상태의 생체 징후 불안정성 측정
- **Lab_Trend:** 입원 기간 내 검사 수치의 호전/악화 방향성 파악
- **Total_Abnormal_Count:** 1.5 STD 기준 통계적 이상치 발생 횟수 산출

**데이터 품질 검증:**
- **결측치 제로:** 병합 후 검사 기록이 있는 환자 중심의 무결성 확보
- **논리성 체크:** 입원 기간 및 나이 데이터의 논리적 오류 검증 완료

**다음 단계:** 03_exploratory_data_analysis.ipynb에서 기본 특성과 심화 패턴 분석을 진행합니다.